# SRQ-FLY D0 — five-task ImageNet-R train-only diagnostic
Run every cell in order on a Colab GPU. This is a five-task diagnostic, not a held-out experiment. It never extracts or evaluates test features. Return the final ZIP for audit.

In [ ]:
# === Edit paths/source only. Do not edit seed, config, representation, or gates. ===
REPO_GIT_URL = 'https://github.com/ZaPhat206/SOHO-CL.git'
REPO_BRANCH = 'feature/srq-fly-d0'
WORK_DIR = '/content/SOHO-CL'
DRIVE_ROOT = '/content/drive/MyDrive/T-SOHO'
DRIVE_TRAIN_CACHE = f'{DRIVE_ROOT}/imagenetr_train_feature_cache_seed2025'
TRAIN_CACHE_DIR = '/content/imagenetr_train_feature_cache_seed2025'
DRIVE_LARGE_WTA_CACHE = f'{DRIVE_ROOT}/tail_fly_imagenetr_wta_cache_seed2025'
LARGE_WTA_CACHE_DIR = '/content/srq_fly_wta_h10000_seed2025'
DRIVE_COMPACT_WTA_CACHE = f'{DRIVE_ROOT}/srq_fly_wta_h4096_seed2025'
COMPACT_WTA_CACHE_DIR = '/content/srq_fly_wta_h4096_seed2025'
OUTPUT_DIR = f'{DRIVE_ROOT}/srq_fly_imagenetr_d0_seed2025'
CHECKPOINT_SOURCE = 'huggingface'  # or 'google_drive'
DRIVE_CHECKPOINT_PATH = f'{DRIVE_ROOT}/model.safetensors'
BATCH_SIZE = 128
SEED = 2025
CHECKPOINT_SIZE = 346284714
CHECKPOINT_SHA256 = '32aa17d6e17b43500f531d5f6dc9bc93e56ed8841b8a75682e1bb295d722405b'
CONFIG_SHA256 = '039e243543c46d8f4ab6984197feccbc4ac4e9d8f96dd6e80f19c49e6460fbd1'

In [ ]:
# Runtime setup. chdir first so replacing an old clone cannot invalidate cwd.
from google.colab import drive
drive.mount('/content/drive')
import hashlib, json, os, shutil, subprocess, sys, time
from pathlib import Path
import torch
assert torch.cuda.is_available(), 'Select Runtime > Change runtime type > T4 GPU.'
os.chdir('/content')
repo_path = Path(WORK_DIR)
if repo_path.exists(): shutil.rmtree(repo_path)
clone = subprocess.run(['git', 'clone', '--branch', REPO_BRANCH, '--single-branch', REPO_GIT_URL, WORK_DIR], text=True, capture_output=True)
print(clone.stdout, clone.stderr, sep='')
assert clone.returncode == 0, f'Clone failed ({clone.returncode}). Confirm the branch was pushed.'
os.chdir(WORK_DIR)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-kaggle.txt', 'kagglehub', 'huggingface_hub'], check=True)
commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
config_path = Path('configs/srq_fly_imagenetr_d0_train_only.json')
assert hashlib.sha256(config_path.read_bytes()).hexdigest() == CONFIG_SHA256, 'Locked config identity mismatch.'
print('repo commit:', commit)
print('GPU:', torch.cuda.get_device_name(0))
print('locked seed:', SEED, '| config SHA-256:', CONFIG_SHA256)

In [ ]:
# Obtain and verify the exact frozen ViT checkpoint.
if CHECKPOINT_SOURCE == 'huggingface':
    from huggingface_hub import hf_hub_download
    CHECKPOINT_PATH = hf_hub_download('timm/vit_base_patch16_224.augreg2_in21k_ft_in1k', 'model.safetensors')
elif CHECKPOINT_SOURCE == 'google_drive':
    CHECKPOINT_PATH = DRIVE_CHECKPOINT_PATH
else:
    raise ValueError('CHECKPOINT_SOURCE must be huggingface or google_drive')
checkpoint = Path(CHECKPOINT_PATH)
digest = hashlib.sha256(checkpoint.read_bytes()).hexdigest()
assert checkpoint.stat().st_size == CHECKPOINT_SIZE and digest == CHECKPOINT_SHA256
print('checkpoint PASS:', checkpoint, '| SHA-256:', digest)

In [ ]:
# Resolve processed ImageNet-R. Test paths are indexed only to verify mapping.
import kagglehub
from torchvision.datasets import ImageFolder
download_root = Path(kagglehub.dataset_download('zaphat206/imagenet-r')).resolve()
directories = [download_root] + [p for p in download_root.rglob('*') if p.is_dir()]
matches = sorted({p.resolve() for p in directories if (p/'train').is_dir() and (p/'test').is_dir()})
assert len(matches) == 1, f'Expected one processed root, found: {matches}'
image_root = matches[0]
processed_root = Path('/content/processed_datasets')
loader_link = processed_root/'imagenet-r'
processed_root.mkdir(parents=True, exist_ok=True)
if loader_link.is_symlink() or loader_link.is_file(): loader_link.unlink()
elif loader_link.exists(): shutil.rmtree(loader_link)
loader_link.symlink_to(image_root, target_is_directory=True)
train_index, test_index = ImageFolder(image_root/'train'), ImageFolder(image_root/'test')
assert len(train_index.classes) == len(test_index.classes) == 200
assert train_index.class_to_idx == test_index.class_to_idx
print('dataset:', image_root, '| train/test:', len(train_index), len(test_index), '| classes: 200')

In [ ]:
# Q0 correctness gate: synthetic data only.
tests = ['tests/test_srq_fly_math.py', 'tests/test_srq_fly_learner.py', 'tests/test_srq_fly_d0.py']
subprocess.run([sys.executable, '-m', 'pytest', '-q', *tests], check=True)
print('SRQ-FLY Q0 correctness gate: PASS')

In [ ]:
# Restore or extract TRAIN embeddings only; every copy/extraction stage prints progress.
local_cache, drive_cache = Path(TRAIN_CACHE_DIR), Path(DRIVE_TRAIN_CACHE)
if not (local_cache/'metadata.json').is_file():
    if (drive_cache/'metadata.json').is_file():
        print('Restoring train-only feature cache from Drive...', flush=True)
        local_cache.mkdir(parents=True, exist_ok=True)
        for source in sorted(drive_cache.iterdir()):
            print('COPY', source.name, f'{source.stat().st_size/2**20:.1f} MiB', flush=True)
            shutil.copy2(source, local_cache/source.name)
    else:
        command = [sys.executable, '-u', 'tools/experiment_runner.py', '--extract-features-only', '--extract-train-only', '--root', str(processed_root), '--backbone-checkpoint', CHECKPOINT_PATH, '--backbone-checkpoint-size', str(CHECKPOINT_SIZE), '--backbone-checkpoint-sha256', CHECKPOINT_SHA256, '--feature-cache-dir', TRAIN_CACHE_DIR, '--output-dir', '/content/srq_fly_imagenetr_extract', '--dataset', 'ImageNet-R', '--model-name', 'vit_base_patch16_224', '--data-augmentation', 'vit', '--seed', str(SEED), '--num-classes', '200', '--num-tasks', '20', '--device', 'cuda', '--batch-size', str(BATCH_SIZE), '--num-workers', '2']
        print('Starting 20-task TRAIN-only ViT extraction. Follow task/tqdm lines.', flush=True)
        subprocess.run(command, check=True)
        assert not (local_cache/'test.pt').exists()
        assert not drive_cache.exists(), 'Incomplete Drive feature cache exists; inspect it first.'
        drive_cache.mkdir(parents=True)
        for source in sorted(local_cache.iterdir()):
            print('SAVE', source.name, 'to Drive', flush=True); shutil.copy2(source, drive_cache/source.name)
metadata = json.loads((local_cache/'metadata.json').read_text())
assert metadata['dataset'] == 'ImageNet-R' and metadata['checkpoint_sha256'] == CHECKPOINT_SHA256
assert metadata['feature_dim'] == 768 and metadata['finite'] is True
assert metadata['test_features_materialized'] is False and not (local_cache/'test.pt').exists()
print('train feature cache PASS:', metadata['train_shape'], '| test.pt absent')

In [ ]:
# Restore reusable WTA experiment caches. The 10k cache normally already exists from Tail/CertiFLY.
def restore_cache(drive_path, local_path):
    source, target = Path(drive_path), Path(local_path)
    if not (target/'metadata.json').is_file() and (source/'metadata.json').is_file():
        print('RESTORE', source, flush=True); target.mkdir(parents=True, exist_ok=True)
        for item in sorted(source.iterdir()):
            print('COPY', item.name, f'{item.stat().st_size/2**20:.1f} MiB', flush=True); shutil.copy2(item, target/item.name)
    return (target/'metadata.json').is_file()
large_restored = restore_cache(DRIVE_LARGE_WTA_CACHE, LARGE_WTA_CACHE_DIR)
compact_restored = restore_cache(DRIVE_COMPACT_WTA_CACHE, COMPACT_WTA_CACHE_DIR)
print('10k WTA:', 'restored' if large_restored else 'will build', '| 4096 WTA:', 'restored' if compact_restored else 'will build')

In [ ]:
# Locked five-task D0. WTA CACHE and START/TASK/DONE/RESUME lines show live progress.
output_path = Path(OUTPUT_DIR)
output_path.mkdir(parents=True, exist_ok=True)
shutil.copy2(config_path, output_path/'locked_config.json')
command = [sys.executable, '-u', 'tools/srq_fly_d0.py', '--config', str(config_path), '--feature-cache-dir', TRAIN_CACHE_DIR, '--large-code-cache-dir', LARGE_WTA_CACHE_DIR, '--compact-code-cache-dir', COMPACT_WTA_CACHE_DIR, '--output-dir', OUTPUT_DIR, '--device', 'cuda', '--require-test-hidden']
print('Starting SRQ-FLY D0: 6 methods x 5 train-validation tasks.', flush=True)
print('WTA CACHE=cache; START/DONE=method; TASK=completed stage; RESUME=already complete.', flush=True)
started = time.time()
completed = subprocess.run(command)
print(f'Runner elapsed: {(time.time()-started)/60:.1f} minutes', flush=True)
assert completed.returncode == 0, 'D0 failed; send the complete traceback without editing config.'
assert (output_path/'d0_results.json').is_file() and not (local_cache/'test.pt').exists()
print('SRQ-FLY D0 process: COMPLETE')

In [ ]:
# Save newly built WTA caches, summarize, download evidence, then STOP.
def save_new_cache(local_path, drive_path, was_restored):
    source, target = Path(local_path), Path(drive_path)
    if not was_restored and (source/'metadata.json').is_file() and not (target/'metadata.json').is_file():
        assert not target.exists(), f'Incomplete Drive cache exists: {target}'
        target.mkdir(parents=True)
        for item in sorted(source.iterdir()):
            print('SAVE', item.name, f'{item.stat().st_size/2**20:.1f} MiB', flush=True); shutil.copy2(item, target/item.name)
save_new_cache(LARGE_WTA_CACHE_DIR, DRIVE_LARGE_WTA_CACHE, large_restored)
save_new_cache(COMPACT_WTA_CACHE_DIR, DRIVE_COMPACT_WTA_CACHE, compact_restored)
import pandas as pd
result = json.loads((output_path/'d0_results.json').read_text())
rows = [{'method':x['method'], 'status':x['status'], 'validation_AA':x.get('validation_average_accuracy'), 'persistent_state_bytes':x.get('persistent_state_bytes'), 'max_solver_residual':x.get('maximum_solver_relative_residual')} for x in result['results']]
display(pd.DataFrame(rows).sort_values('validation_AA', ascending=False))
print('decision:', result['status'])
print('gates:', json.dumps(result['gates'], indent=2))
archive = shutil.make_archive('/content/srq_fly_imagenetr_d0_train_only', 'zip', root_dir=OUTPUT_DIR)
print('artifact SHA-256:', hashlib.sha256(Path(archive).read_bytes()).hexdigest())
from google.colab import files
files.download(archive)
print('STOP. Send the ZIP for audit; do not evaluate ImageNet-R test.')